# Agent的高级用法-错误处理机制
## 情况1：设置True/false/固定字符串
举例1：handle_errors为true

In [1]:
import os
from enum import Enum
from typing import Optional
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)

In [3]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy,MultipleStructuredOutputsError,StructuredOutputValidationError
from rich import print as rprint
def custom_error_handler(error:Exception) -> str:
    """自定义错误处理器"""
    error_str = str(error)
    print(f"捕获到的错误类型是：{type(error).__name__}")
    print(f"错误详情信息{error_str}")
    if isinstance(error, MultipleStructuredOutputsError):
        return"检测到多个响应，请选择一个进行返回"
    elif isinstance(error,StructuredOutputValidationError):
        return"数据格式有误，请检查字段是否符合要求"
    else:
        return f"Error:{error_str}"
class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")
class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")
agent = create_agent(
        model=model,
    response_format=ToolStrategy(
    Union[ContactInfo, EventDetails],
    tool_message_content="提取完成！",
    handle_errors=custom_error_handler,
#handle_errors="请检查输入数据"
)
)
result = agent.invoke({
"messages": [{
"role": "user",
"content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
}]
})
rprint(result)
# for msg in result["messages"]:
# msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

捕获到的错误类型是：MultipleStructuredOutputsError
错误详情信息Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.


{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='077fbed2-afdd-4620-abea-0c802df65a47'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 112,
                    'prompt_tokens': 411,
                    'total_tokens': 523,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 27
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'f06ec298-03cc-4f3e-bc12-edebeb9360ba',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a07ad5-7fbf-71b3-b1bf-17a7b415526b-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_00_mNUgGiSqxPsNJyhb0OYA7589',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_01_B2CbdcXNSgcxTmTRaBVG1711',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 411,
                'output_tokens': 112,
                'total_tokens': 523,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='检测到多个响应，请选择一个进行返回',
            name='ContactInfo',
            id='1e331cdb-fdff-40dd-92e2-39aced607fac',
            tool_call_id='call_00_mNUgGiSqxPsNJyhb0OYA7589'
        ),
        ToolMessage(
            content='检测到多个响应，请选择一个进行返回',
            name='EventDetails',
            id='5c13aef6-cbab-4a55-9e52-9ccc505489b5',
            tool_call_id='call_01_B2CbdcXNSgcxTmTRaBVG1711'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 59,
                    'prompt_tokens': 568,
                    'total_tokens': 627,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                    'prompt_cache_hit_tokens': 512,
                    'prompt_cache_miss_tokens': 56
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'd4c17dc0-eb6c-49a9-8669-fac393046a5d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a07ad5-86b3-7923-ad39-3a1716f9fadd-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_00_qp1imD1dh2fn9sbLT8CY1933',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 568,
                'output_tokens': 59,
                'total_tokens': 627,
                'input_token_details': {'cache_read': 512},
            